In [8]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Environment Configuration
# Load variables with override enabled to ensure fresh credentials
load_dotenv(override=True)

# 2. Database Connection Setup
# Constructing the connection string and initializing the SQLAlchemy engine
DATABASE_URL = f"postgresql://{os.getenv('user')}:{os.getenv('password')}@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
engine = create_engine(DATABASE_URL)

# 3. Schema Definition
# Exact SQL table schema definition (maintaining specific order and naming)
cols_sql = [
    'ticker', 'asset_name', 'asset_type', 'sector', 'exchange', 
    'currency', 'beta', 'volatility', 'max_drawdown', 'risk_category', 
    'investment_style', 'dividend_yield', 'avg_return_1y', 
    'avg_return_3y', 'avg_return_5y'
]

# 4. Data Acquisition
# Load source CSV and normalize column names to lowercase for consistency
df = pd.read_csv('../1_data_extraction/data/assets_info.csv')
df.columns = df.columns.str.lower()

# 5. Data Alignment & Reindexing
# Align the DataFrame with the target database schema.
# Note: Missing columns (e.g., 'avg_return_5y') will be initialized with NaNs.
df = df.reindex(columns=cols_sql)

# 6. Integrity Cleaning
# Prevent 'NotNullViolation' by removing rows where the mandatory 'asset_name' is missing
df = df.dropna(subset=['asset_name'])

# 7. Data Loading (ETL Final Stage)
try:
    df.to_sql(
        name='assets', 
        con=engine, 
        if_exists='append', 
        index=False, 
        method='multi', 
        chunksize=500
    )
    print(f"Success! {len(df)} assets have been successfully loaded.")
except Exception as e:
    print(f"Database Integrity Error: {e}")

Success! 49 assets have been successfully loaded.
